# Paper 5, RB-3 - intact model, shuffled rotations and nulls, at single-head level

**Naming.** W1-W7 = September runs behind the paper. RB = November rebuttal runs: RB-0 = single-head run (files
`w8_*`; canonical = the cuDNN 9.10.2 re-run of 4 Nov), RB-1/RB-1b = drift checks, RB-2 = single-head axis map
(5 Nov). This is **RB-3**; it writes to `./rb3_output/`.

**What it answers.** §4.5, Figure 5b and Table 18 rest on three tests of the "top rebuilder", each done on head
*indices*: (1) after the 0-5 window it is the costliest head for top-1; (2) in the intact model the same head costs
little; (3) giving it rotations for shuffled positions costs about as much as removing them. RB-0 already gave (1)
for every single (block, head). RB-3 completes the single-head version of all three, with heads chosen by the rule
RB-2 fixed before seeing any of this (`ssdc_b11`: SSDC lost entering block 11, blocks 6-10):

| Part | What | Data | Models |
|---|---|---|---|
| A | **intact model**: every single (block, head) in 6-11 kept without rotations, nothing else changed | ILSVRC 5,000 | naver, ViT-S, reg1-gap |
| B | **shuffled rotations** for the top 4 single heads (by `ssdc_b11`), after the window and in the intact model, with both W7 permutations | ILSVRC 5,000 | same |
| C | **Table 18 / Table 10 references under cuDNN 9.10.2**: the W7 arms RB-0 did not repeat (all-shuffled ×2, window-shuffled, top index group shuffled in the intact model) and top-1 with prefix attention blocked | ILSVRC 5,000 | same |
| D | **index-grouping null with the right rule**: the best head per block and the six best heads by `ssdc_b11`, against RB-0's random one-head-per-block sets | repack 1,000 + ILSVRC 5,000 | + DINOv3 (SSDC only) |

Part C exists because W7's ViT-S numbers were made with cuDNN 9.26-like numerics (RB-1b), while W5 and everything
since use 9.10.2. On naver and reg1-gap these arms must reproduce W7 bit for bit (RB-0 showed W7 = 9.10.2 there);
on ViT-S they give the 9.10.2 values for the revised Table 18 and Table 10.

**Coherence.** Same env (`env_paper5_cudnn910`), datasets, image counts, batch, seed, fp32 and code paths as RB-0.
Gates demand bit-identical agreement with the canonical RB-0 (references, every arm RB-0 already ran, the random
sets) and with W7 on naver and reg1-gap. Runtime ~45-60 min on one H100.

In [1]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "datasets", "scipy", "pandas"]:
    ensure(p)
print("environment ready")

/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


environment ready


In [2]:
import os, json, time, warnings, traceback, shutil, hashlib, platform, socket
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
warnings.filterwarnings("ignore", category=UserWarning)
print("python", sys.executable)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f"memory: {free/1e9:.1f} / {total/1e9:.1f} GB free")

python /workspace/NRDL/env_paper5_cudnn910/bin/python
torch 2.10.0+cu128 | timm 1.0.29 | cuda True


device: NVIDIA H100 80GB HBM3
memory: 84.5 / 85.0 GB free


In [3]:
# config
REPO_PATH  = os.environ.get("RB3_REPO", "")        # .../vit-sae-analysis/project_code/src ("" = search)
RB0_DIR    = os.environ.get("RB3_RB0_DIR", "")     # the canonical RB-0 folder (cuDNN 9.10.2 re-run, 4 Nov)
MODELS_RUN = [m for m in os.environ.get("RB3_MODELS", "naver,DINOv3,small,reg1_gap").split(",") if m]
RESUME     = os.environ.get("RB3_RESUME", "1") == "1"
STRICT     = os.environ.get("RB3_STRICT", "1") == "1"   # 1: a model whose reference gates fail is not continued
TEST_MODE  = os.environ.get("RB3_TEST", "0") == "1"     # CPU smoke test on tiny random models, not for results
OUT_DIR    = Path(os.environ.get("RB3_OUT", "rb3_output"))

N_SSDC, N_ACC, BATCH_SIZE = 1000, 5000, 256          # W5 / W7 / RB-0, unchanged
METRIC, TORCH_SEED, INPUT_SIZE = "manhattan", 20260908, 224
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"  # every W2-W6, RB-0 and RB-2 SSDC number
ACC_DATASET  = "ILSVRC/imagenet-1k"                  # W7 and RB-0 top-1 (Table 18)

N_BLOCKS = 12
WINDOW   = list(range(0, 6))
POST     = list(range(6, 12))
FINAL    = 12
# probe settings are not used by RB-3 but measure()'s cell defines the probe helpers
PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC = 5e-3, 20, 512, 0.8
PROBE_CHUNK, PROBE_DEVICE = 8192, None
NOISE_SEEDS, NOISE_BOOT = [0, 1, 2], 200

EXACT_TOL = 0.0
EXPECT_TORCH, EXPECT_CUDNN = "2.10.0+cu128", 91002
EXPECT_CUDNN_PKGS = {"nvidia-cudnn-cu12": "9.10.2.21"}
FIRST_LAYER_910 = {"naver": 0.00041303808169361526, "DINOv3": 0.0004745740108653513,
                   "small": 0.000510305810057617, "reg1_gap": 0.00026782853342761795}
RB0_SHA256 = {
    "w8_partA_naver.json":     "f50b694d24279ce63b8dac8da1906b7e60ab6522ce8de4c992b1c87ceed10914",
    "w8_partA_DINOv3.json":    "f1d963652ac3867c79bb170da2b2dd86ff8e978008f5ff59304a69947bb909b5",
    "w8_partA_small.json":     "be01ed6297ef819a8fc219853c07fc9a81a76d2c6ffbe9da8501379fdf6acbbb",
    "w8_partA_reg1_gap.json":  "da5ae49a2abc21aa0df8f4c43b249727a7edfe444254e239e637ed96a312fefb",
    "w8_partC_naver.json":     "c983414404ca645bc8144a2bd4ec76bfb583f4c5c9588ec8faffe853e5c3b985",
    "w8_partC_small.json":     "bf895274486dfaa9ab042118e15bb7c9863a5f6201ab27ec11fb539ecf7126f6",
    "w8_partC_reg1_gap.json":  "8b5641235e05b20bfdc597032097b2f85878c228bdc32ec8b63888227e393698",
    "w8_partD_naver.json":     "4b385c6bfde78d76a51276108fd9de741a394db3ccdb56980018a8ac1a885836",
    "w8_partD_DINOv3.json":    "6a49100079cfe304f1e111b671d11f61c1c4763d55280f0b37c18dfd768c11c0",
    "w8_partD_small.json":     "5de620a0ea29fa08c29dc56c87e45baabdf111781c79bf69191180e664e29fd0",
    "w8_partD_reg1_gap.json":  "85d72d98fff492d5ff72955169e4739e5f0b8b96d73c11ffb794f9f02ed68362",
    "w8_perimage_naver.npz":   "5fe4a53ecbf99bc583c56ef6ec9f774c5bd6027e1734dce16fe0c230c36358c6",
    "w8_perimage_small.npz":   "b9f6a3833766b73618c82b3e5ecbb81af672b11eaa49fab503e27bb8734cf40b",
    "w8_perimage_reg1_gap.npz": "10310f0c0109d65f0b594f7774006ddd2c69f5fc74438b13d32956d39738643e",
}
MODEL_SPECS = {
    "naver":    dict(name="vit_base_patch16_rope_224.naver_in1k",        classifier=True,  label="naver ViT-B"),
    "DINOv3":   dict(name="vit_base_patch16_dinov3.lvd1689m",            classifier=False, label="DINOv3 ViT-B"),
    "small":    dict(name="vit_small_patch16_rope_224.naver_in1k",       classifier=True,  label="naver ViT-S"),
    "reg1_gap": dict(name="vit_base_patch16_rope_reg1_gap_256.sbb_in1k", classifier=True,  label="reg1-gap ViT-B"),
}
RB0_FILES = {k: [f"w8_partA_{k}.json", f"w8_partD_{k}.json"] +
                ([f"w8_partC_{k}.json", f"w8_perimage_{k}.npz"] if s["classifier"] else [])
             for k, s in MODEL_SPECS.items()}

# rules, fixed before the run (RB-2's pre-registered decision: ssdc_b11)
RULE_RANK = ("single heads in blocks 6-10 ranked by SSDC lost entering block 11 vs Cref (RB-0 part A); block-11 heads "
             "ranked separately by SSDC lost at the output of block 11. This is RB-2's pre-registered decision (ssdc_b11).")
TOP_SHUFFLE = 4      # part B: top 4 heads by the rule (RB-0 used 3 for its intact checks; 4 covers near-ties)
RULE_BEST_PER_BLOCK = "blocks 6-10: the head with the largest ssdc_b11 cost; block 11: the head with the largest out11 cost"
RULE_TOP6 = "the six heads with the largest ssdc_b11 cost in blocks 6-10"
# RB-2's top 5 by ssdc_b11 (recomputed from RB-0 part A and gated)
EXPECT_TOP = {"naver": [(6, 6), (7, 2), (6, 1), (6, 2), (8, 10)], "DINOv3": [(6, 2), (9, 10), (7, 9), (10, 0), (10, 8)],
              "small": [(7, 2), (7, 4), (6, 0), (7, 3), (8, 4)], "reg1_gap": [(6, 9), (7, 7), (6, 7), (6, 4), (7, 6)]}
W7_PERM_SEED = 7     # W7: g = Generator().manual_seed(7); perms = [randperm(196, g) for _ in range(2)]; RB-0 used perms[0]
N_RAND_GATE = 2      # RB-0 part D random sets re-measured as a reproducibility gate
REPLAY_CHECK_N = 512

if TEST_MODE:
    N_SSDC, N_ACC, BATCH_SIZE = 24, 32, 8
    PROBE_PASSES, NOISE_BOOT = 2, 10
    OUT_DIR = Path(os.environ.get("RB3_OUT", "rb3_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)
_bad = [m for m in MODELS_RUN if m not in MODEL_SPECS]
if _bad:
    raise SystemExit(f"Unknown entries in RB3_MODELS: {_bad}")

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDEV = PROBE_DEVICE or DEVICE
GATES = []
GATE_FILE = OUT_DIR / f"rb3_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))
    return bool(passed) or skip
print(f"device {DEVICE} | fp32 | models {MODELS_RUN}")
print(f"SSDC: {SSDC_DATASET} x {N_SSDC} | top-1: {ACC_DATASET} x {N_ACC} | batch {BATCH_SIZE} | seed {TORCH_SEED}")
print(f"output {OUT_DIR.resolve()} | resume {RESUME} | strict {STRICT} | test mode {TEST_MODE} | run {RUN_STAMP}")

device cuda | fp32 | models ['naver', 'DINOv3', 'small', 'reg1_gap']
SSDC: benjamin-paine/imagenet-1k-256x256 x 1000 | top-1: ILSVRC/imagenet-1k x 5000 | batch 256 | seed 20260908
output /workspace/NRDL/experiments/rb3/rb3_output | resume True | strict True | test mode False | run 2026-11-05T10-07-19Z


## Pre-flight: are we in `env_paper5_cudnn910`?
A small GPU convolution loads cuDNN; then the notebook checks the version PyTorch reports, which library files were
actually loaded (from the process's own memory map), the installed cuDNN packages, and that the TF32 / cuDNN flags
are PyTorch's defaults (W5 and RB-0 ran with defaults). Nothing is measured if any check fails.

In [4]:
def _sh(cmd, timeout=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return ((r.stdout or "") + (r.stderr or "")).strip()
    except Exception as ex:
        return f"(failed: {ex!r})"

def _loaded_cudnn():
    try:
        with open("/proc/self/maps") as f:
            return sorted({l.split()[-1] for l in f if "libcudnn" in l and l.split()[-1].startswith("/")})
    except Exception:
        return []

import importlib.metadata as _md
if DEVICE == "cuda":
    _x = torch.randn(2, 3, 32, 32, device="cuda"); _w = torch.randn(8, 3, 16, 16, device="cuda")
    F.conv2d(_x, _w, stride=16); torch.cuda.synchronize(); del _x, _w
CUDNN_PKGS = {}
for _d in _md.distributions():
    _n = (_d.metadata.get("Name") or "").lower()
    if "cudnn" in _n: CUDNN_PKGS[_n] = _d.version
ENV = {
    "python": sys.executable, "prefix": sys.prefix, "host": socket.gethostname(), "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__,
    "cudnn_reported": torch.backends.cudnn.version() if DEVICE == "cuda" else None,
    "libcudnn_loaded": _loaded_cudnn(), "cudnn_packages": CUDNN_PKGS,
    "flags": {"matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32, "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
              "cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic,
              "float32_matmul_precision": torch.get_float32_matmul_precision()},
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "gpu": _sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,memory.used,memory.total --format=csv") if DEVICE == "cuda" else "cpu",
}
(OUT_DIR / f"rb3_pip_freeze_{RUN_STAMP}.txt").write_text(_sh(f"{sys.executable} -m pip freeze", timeout=300))
(OUT_DIR / f"rb3_environment_{RUN_STAMP}.json").write_text(json.dumps(ENV, indent=1, default=str))
print(json.dumps({k: v for k, v in ENV.items() if k != "gpu"}, indent=1, default=str)); print(ENV["gpu"])

_pre = []
if TEST_MODE:
    gate("pre-flight (cuDNN 9.10.2 environment)", "env", True, "test mode on CPU", skip=True)
else:
    _pre.append(gate("CUDA available", "env", DEVICE == "cuda"))
    _pre.append(gate(f"torch is {EXPECT_TORCH}", "env", torch.__version__ == EXPECT_TORCH, torch.__version__))
    _pre.append(gate(f"PyTorch reports cuDNN {EXPECT_CUDNN} (9.10.2)", "env", ENV["cudnn_reported"] == EXPECT_CUDNN,
                     str(ENV["cudnn_reported"])))
    _prefix = os.path.realpath(sys.prefix)
    _libs = ENV["libcudnn_loaded"]
    _pre.append(gate("every loaded libcudnn file comes from this environment's nvidia/cudnn folder", "env",
                     bool(_libs) and all(os.path.realpath(l).startswith(_prefix) and "/nvidia/cudnn/" in l for l in _libs),
                     "; ".join(Path(l).name for l in _libs) + f" (prefix {_prefix})"))
    _pre.append(gate("the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13)", "env",
                     CUDNN_PKGS == EXPECT_CUDNN_PKGS, str(CUDNN_PKGS)))
    _f = ENV["flags"]
    _pre.append(gate("TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0)", "env",
                     _f["matmul_allow_tf32"] is False and _f["cudnn_allow_tf32"] is True and _f["cudnn_benchmark"] is False
                     and _f["cudnn_deterministic"] is False and _f["float32_matmul_precision"] == "highest", str(_f)))
PREFLIGHT_OK = all(_pre)
if not PREFLIGHT_OK:
    raise SystemExit("PRE-FLIGHT FAILED: this notebook must run in env_paper5_cudnn910 (cuDNN 9.10.2), with PyTorch's "
                     "default flags. Nothing was measured. Switch the kernel / python and run again.")
print("\npre-flight OK")

{
 "python": "/workspace/NRDL/env_paper5_cudnn910/bin/python",
 "prefix": "/workspace/NRDL/env_paper5_cudnn910",
 "host": "gpu-pod",
 "platform": "Linux-6.8.0-41-generic-x86_64-with-glibc2.35",
 "torch": "2.10.0+cu128",
 "timm": "1.0.29",
 "numpy": "2.4.6",
 "cudnn_reported": 91002,
 "libcudnn_loaded": [
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_engines_precompiled.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_engines_runtime_compiled.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_graph.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_heuristic.so.9"
 ],
 "cudnn_packages": {
  "nvidia-cudnn-cu12": "9.10.2.21"
 },
 "flags": {
  "matmul_allow_tf32": false,
  "cudnn_allow_tf32": t

  [PASS] env: torch is 2.10.0+cu128: 2.10.0+cu128


  [PASS] env: PyTorch reports cuDNN 91002 (9.10.2): 91002


  [PASS] env: every loaded libcudnn file comes from this environment's nvidia/cudnn folder: libcudnn.so.9; libcudnn_engines_precompiled.so.9; libcudnn_engines_runtime_compiled.so.9; libcudnn_graph.so.9; libcudnn_heuristic.so.9 (prefix /workspace/NRDL/env_paper5_cudnn910)


  [PASS] env: the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13): {'nvidia-cudnn-cu12': '9.10.2.21'}


  [PASS] env: TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0): {'matmul_allow_tf32': False, 'cudnn_allow_tf32': True, 'cudnn_benchmark': False, 'cudnn_deterministic': False, 'float32_matmul_precision': 'highest'}



pre-flight OK


In [5]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB3_REPO. Tried: " +
                            ", ".join(str(c) for c in cands[:8]))

repo = resolve_repo()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
REPO_COMMIT = _sh(f"git -C {repo} rev-parse HEAD")
print("repo root:", repo, "| commit", REPO_COMMIT)

from metrics.ssdc import (evaluate_ssdc, spatial_similarity_distance_correlation, SSDCAccumulator,
                          _register_ssdc_accumulator_hooks)
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope
from main.model import predict
from main.prep_data import prep_data
from experiments.common import load_imagenet

from scipy.spatial.distance import cdist as _cdist
from scipy.stats import rankdata
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
_r = spatial_similarity_distance_correlation(np.random.default_rng(0).normal(size=(_G * _G,) * 2),
                                             grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric",
     _a > 0.999 and _b < -0.999 and abs(_r) < 0.1, f"{_a:+.3f}/{_b:+.3f}/{_r:+.3f}")

repo root: /workspace/NRDL/clone/vit-sae-analysis/project_code/src | commit 28e0856cd87c8582aff2783368b3d2f098ae3190


  [PASS] repo-metric: imported metric satisfies its unit test: +1.000/-1.000/-0.005


True

## The canonical RB-0 (fingerprint check)
RB-3 is gated against RB-0 parts A, C and D and reuses their numbers, so it must read the **cuDNN 9.10.2 re-run**
(`rb0_cudnn910/w8_output`). Every file it reads is checked by sha256.

In [6]:
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

RB0 = {}
if not RB0_DIR and not TEST_MODE:
    raise SystemExit("Set RB3_RB0_DIR to the canonical RB-0 folder: "
                     "/workspace/NRDL/experiments/rb0_cudnn910/w8_output")
_rb0 = Path(RB0_DIR).expanduser() if RB0_DIR else None
_rb0_bad = []
for key in MODELS_RUN:
    for fn in RB0_FILES[key]:
        p = (_rb0 / fn) if _rb0 else None
        if p is None or not p.exists():
            if TEST_MODE: gate(f"RB-0 file {fn} present", key, True, "test mode: no RB-0 folder", skip=True)
            else: gate(f"RB-0 file {fn} present", key, False, str(p)); _rb0_bad.append(fn)
            continue
        if TEST_MODE:
            gate(f"{fn} is the canonical RB-0 file", key, True, "test mode: hash not checked", skip=True)
        else:
            d = sha256_file(p)
            if not gate(f"{fn} is the canonical RB-0 file (cuDNN 9.10.2 re-run)", key, d == RB0_SHA256[fn],
                        f"sha256 {d[:16]}... (expected {RB0_SHA256[fn][:16]}...)"):
                _rb0_bad.append(fn)
if _rb0_bad:
    raise SystemExit(f"RB-0 FINGERPRINT FAILED for {_rb0_bad}. RB3_RB0_DIR must be the cuDNN 9.10.2 re-run "
                     "(rb0_cudnn910/w8_output, 4 Nov). Nothing was measured.")
for key in MODELS_RUN:
    d = {}
    for part in "ACD":
        p = (_rb0 / f"w8_part{part}_{key}.json") if _rb0 else None
        d[part] = json.loads(p.read_text()) if (p is not None and p.exists()) else None
    p = (_rb0 / f"w8_perimage_{key}.npz") if _rb0 else None
    d["P"] = ({n: z_[n] for z_ in [np.load(p)] for n in z_.files} if (p is not None and p.exists()) else None)
    RB0[key] = d
    need = "ACD" if MODEL_SPECS[key]["classifier"] else "AD"
    gate("RB-0 parts needed by RB-3 are present and complete", key,
         all((d[x] or {}).get("complete") for x in need) and (d["P"] is not None or not MODEL_SPECS[key]["classifier"]),
         ", ".join(f"{x}={'complete' if (d[x] or {}).get('complete') else 'missing'}" for x in need))

# the pre-registered selection, recomputed from RB-0 part A
def rank_ssdc_b11(RA):
    c = RA["conditions"]; nh = RA["n_heads"]; cref = c["Cref"]["ssdc"]
    r = sorted([(cref[11] - c[f"s_b{k}_h{h}"]["ssdc"][11], k, h) for k in POST[:-1] for h in range(nh)], key=lambda t: (-t[0], t[1], t[2]))
    r11 = sorted([(cref[FINAL] - c[f"s_b11_h{h}"]["ssdc"][FINAL], 11, h) for h in range(nh)], key=lambda t: (-t[0], t[2]))
    return [(k, h) for _, k, h in r], [(k, h) for _, k, h in r11], {(k, h): v for v, k, h in r + r11}
SEL = {}
for key in MODELS_RUN:
    if not RB0[key]["A"]: continue
    r610, r11, cost = rank_ssdc_b11(RB0[key]["A"])
    best = {k: [max(range(RB0[key]["A"]["n_heads"]), key=lambda h: (cost[(k, h)], -h))] for k in POST[:-1]}
    best[11] = [r11[0][1]]
    top6 = {}
    for k, h in r610[:6]: top6.setdefault(k, []).append(h)
    SEL[key] = {"rank_6_10": r610, "rank_11": r11, "cost": cost, "top": r610[:TOP_SHUFFLE], "best_per_block": best, "top6": top6}
    if TEST_MODE:
        gate("pre-registered top heads equal RB-2's printed ranking", key, True, f"test mode; top {r610[:5]}", skip=True)
    else:
        gate("pre-registered top heads (ssdc_b11, from RB-0 A) equal RB-2's printed ranking", key,
             [tuple(x) for x in r610[:5]] == EXPECT_TOP[key], f"{r610[:5]}")
    print(f"  {key}: top {TOP_SHUFFLE} {r610[:TOP_SHUFFLE]} | best per block {best} | top6 {top6}")

  [PASS] naver: w8_partA_naver.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 f50b694d24279ce6... (expected f50b694d24279ce6...)


  [PASS] naver: w8_partD_naver.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 4b385c6bfde78d76... (expected 4b385c6bfde78d76...)


  [PASS] naver: w8_partC_naver.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 c983414404ca645b... (expected c983414404ca645b...)


  [PASS] naver: w8_perimage_naver.npz is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 5fe4a53ecbf99bc5... (expected 5fe4a53ecbf99bc5...)


  [PASS] DINOv3: w8_partA_DINOv3.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 f1d963652ac3867c... (expected f1d963652ac3867c...)


  [PASS] DINOv3: w8_partD_DINOv3.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 6a49100079cfe304... (expected 6a49100079cfe304...)


... (13 outputs left out)

  naver: top 4 [(6, 6), (7, 2), (6, 1), (6, 2)] | best per block {6: [6], 7: [2], 8: [10], 9: [11], 10: [1], 11: [7]} | top6 {6: [6, 1, 2], 7: [2], 8: [10, 11]}
  [PASS] DINOv3: pre-registered top heads (ssdc_b11, from RB-0 A) equal RB-2's printed ranking: [(6, 2), (9, 10), (7, 9), (10, 0), (10, 8)]


  DINOv3: top 4 [(6, 2), (9, 10), (7, 9), (10, 0)] | best per block {6: [2], 7: [9], 8: [9], 9: [10], 10: [0], 11: [7]} | top6 {6: [2], 9: [10], 7: [9, 1], 10: [0, 8]}
  [PASS] small: pre-registered top heads (ssdc_b11, from RB-0 A) equal RB-2's printed ranking: [(7, 2), (7, 4), (6, 0), (7, 3), (8, 4)]


  small: top 4 [(7, 2), (7, 4), (6, 0), (7, 3)] | best per block {6: [0], 7: [2], 8: [4], 9: [3], 10: [2], 11: [3]} | top6 {7: [2, 4, 3], 6: [0], 8: [4], 9: [3]}
  [PASS] reg1_gap: pre-registered top heads (ssdc_b11, from RB-0 A) equal RB-2's printed ranking: [(6, 9), (7, 7), (6, 7), (6, 4), (7, 6)]


  reg1_gap: top 4 [(6, 9), (7, 7), (6, 7), (6, 4)] | best per block {6: [9], 7: [7], 8: [1], 9: [1], 10: [8], 11: [10]} | top6 {6: [9, 7, 4, 11], 7: [7, 6]}


In [7]:
# committed W5 and W7 anchors (read from the repo's result files)
RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"
def _first(pattern):
    hits = sorted(RES_ROOT.glob(pattern)); return hits[0] if hits else None
W5_FILES = {nm: _first(f"w5/w5_partA_{nm}_*.json") for nm in MODEL_SPECS}
W7_R12 = {"naver": "w7/w7rope_R12_naver_2026-09-24T17-54-52Z.json", "small": "w7/w7rope_R12_small_2026-09-24T22-49-53Z.json",
          "reg1_gap": "w7/w7rope_R12_reg1_gap_MERGED.json"}
W7_C3 = {"naver": "w7/w7rope_C3_naver_2026-09-24T22-49-53Z.json", "small": "w7/w7rope_C3_small_2026-09-24T22-49-53Z.json",
         "reg1_gap": "w7/w7rope_C3_reg1_gap_2026-09-25T01-38-38Z.json"}
HEADLINE_ACC = {"naver": (0.7052, 6, 0.6622), "small": (0.4868, 2, 0.4106), "reg1_gap": (0.5628, 7, 0.5054)}
# cuDNN-insensitive models, where RB-0 reproduced W7 bit for bit; ViT-S is the one that moved (RB-1b)
W7_EXACT = {"naver": True, "small": False, "reg1_gap": True}
ANCH, W7 = {}, {}
if TEST_MODE:
    gate("committed W5 / W7 anchors loaded", "anchors", True, "test mode: tiny random models have no anchors", skip=True)
for nm in ([] if TEST_MODE else MODELS_RUN):
    f = W5_FILES.get(nm)
    if f is None:
        gate("committed W5 file found", nm, False, "missing"); continue
    e = json.load(open(f))
    ANCH[nm] = {"file": f.name, "n_heads": e["n_heads"], "row_planes": e["row_planes"], "col_planes": e["col_planes"],
                "stageC": [d["head"] for d in e["stageC_ranking"]]}
    if not MODEL_SPECS[nm]["classifier"]: continue
    fr, fc = RES_ROOT / W7_R12[nm], RES_ROOT / W7_C3[nm]
    if not (fr.exists() and fc.exists()):
        gate("committed W7 files found", nm, False, f"{fr.name} {fr.exists()} / {fc.name} {fc.exists()}"); continue
    er, ec = json.load(open(fr)), json.load(open(fc))
    acc = {k: v["acc"] for k, v in er["conditions"].items() if "acc" in v}
    pi = RES_ROOT / f"w7/perimage/R12_{nm}_correct.npz"
    W7[nm] = {"acc": acc, "top": er["top_rebuilder"], "prefix_block_acc": ec.get("prefix_block_acc"),
              "img": ({n: z_[n] for z_ in [np.load(pi)] for n in z_.files} if pi.exists() else {})}
    w, top, wt = HEADLINE_ACC[nm]
    gate("committed W7 files match their headline values", nm,
         abs(acc["window_identity"] - w) < 1e-4 and er["top_rebuilder"] == top == ANCH[nm]["stageC"][0]
         and abs(acc[f"win_h{top}"] - wt) < 1e-4 and ec.get("prefix_block_acc") is not None, f"{fr.name}, {fc.name}")
print("W5:", {k: v["file"] for k, v in ANCH.items()}, "| W7:", {k: (v["top"], len(v["img"])) for k, v in W7.items()})

  [PASS] naver: committed W7 files match their headline values: w7rope_R12_naver_2026-09-24T17-54-52Z.json, w7rope_C3_naver_2026-09-24T22-49-53Z.json


  [PASS] small: committed W7 files match their headline values: w7rope_R12_small_2026-09-24T22-49-53Z.json, w7rope_C3_small_2026-09-24T22-49-53Z.json


  [PASS] reg1_gap: committed W7 files match their headline values: w7rope_R12_reg1_gap_MERGED.json, w7rope_C3_reg1_gap_2026-09-25T01-38-38Z.json


W5: {'naver': 'w5_partA_naver_2026-09-13T10-16-41Z.json', 'DINOv3': 'w5_partA_DINOv3_2026-09-13T10-16-41Z.json', 'small': 'w5_partA_small_2026-09-13T10-16-41Z.json', 'reg1_gap': 'w5_partA_reg1_gap_2026-09-13T10-16-41Z.json'} | W7: {'naver': (6, 32), 'small': (2, 20), 'reg1_gap': (7, 6)}


In [8]:
# datasets: explicit ids, no silent fallback
if TEST_MODE:
    from PIL import Image as _PIL
    class _FakeStream:
        def __init__(self, n, seed):
            self.n, self.seed = n, seed
        def __iter__(self):
            rng = np.random.default_rng(self.seed)
            for i in range(self.n):
                arr = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
                yield {"image": _PIL.fromarray(arr), "label": int(i % 10)}
    DS_SSDC = _FakeStream(400, 1); DS_ACC = _FakeStream(400, 2)
    print("TEST MODE: random images stand in for both datasets")
else:
    DS_SSDC = load_imagenet(split="validation", streaming=True, dataset_id=SSDC_DATASET, fallback_id=None)
    DS_ACC = load_imagenet(split="validation", streaming=True, dataset_id=ACC_DATASET, fallback_id=None)
    gate("SSDC dataset is the repack (W2-W6, RB-0, RB-2)", "data", SSDC_DATASET == "benjamin-paine/imagenet-1k-256x256", SSDC_DATASET)
    gate("top-1 dataset is the ILSVRC split (W7, RB-0)", "data", ACC_DATASET == "ILSVRC/imagenet-1k", ACC_DATASET)

loaded benjamin-paine/imagenet-1k-256x256 [validation]


loaded ILSVRC/imagenet-1k [validation]
  [PASS] data: SSDC dataset is the repack (W2-W6, RB-0, RB-2): benjamin-paine/imagenet-1k-256x256


  [PASS] data: top-1 dataset is the ILSVRC split (W7, RB-0): ILSVRC/imagenet-1k


In [9]:
# rotary helpers: from W5 / RB-0
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None

def attn_attrs(model, source):
    rh, src, nh = False, "default(interleaved)", None
    for blk in get_vit_blocks(model, source):
        a = get_block_attention(blk, source)
        for at in ("rotate_half", "half", "rope_rotate_half"):
            v = getattr(a, at, None)
            if isinstance(v, bool): rh, src = v, at; break
        nh = getattr(a, "num_heads", None); break
    return rh, src, nh

def plane_columns(p, n, rh): return [p, p + n] if rh else [2 * p, 2 * p + 1]

def derive_axis(emb, n_planes, rh, H, W):
    sin, cos = emb.detach().float().chunk(2, -1); rows = []; dec = True
    for p in range(n_planes):
        c = plane_columns(p, n_planes, rh)[0]
        z = torch.complex(cos[:, c], sin[:, c]).reshape(H, W)
        ra = (z - z.mean(dim=1, keepdim=True)).abs().mean().item()
        rb = (z - z.mean(dim=0, keepdim=True)).abs().mean().item()
        ax = "row" if ra < rb else "col"; lo, hi = sorted([ra, rb]); r = lo / hi if hi > 1e-12 else 1.0
        if r > 0.10: dec = False
        rows.append({"plane": p, "axis": ax, "ratio": r})
    return rows, [r["plane"] for r in rows if r["axis"] == "row"], [r["plane"] for r in rows if r["axis"] == "col"], dec

def mask_planes(emb, kill, n, rh):
    if not kill: return emb
    s, c = emb.chunk(2, -1); s, c = s.clone(), c.clone()
    for p in kill:
        for col in plane_columns(p, n, rh): s[..., col] = 0.0; c[..., col] = 1.0
    return torch.cat([s, c], -1)

def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
IDEN = identity_like

def per_head(heads, inner, n_heads):
    hs = sorted(set(heads))
    def tf(emb):
        if emb.dim() != 2:
            raise RuntimeError(f"per_head expected 2-D rope, got {tuple(emb.shape)}: stacked overrides")
        out = emb.unsqueeze(0).repeat(n_heads, 1, 1)
        if hs:
            t = inner(emb)
            for h in hs: out[h] = t
        return out
    return tf

class RopeComposite:
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source
        s.segs = [(list(b), t) for b, t in segs]; seen = set()
        for b, _ in s.segs:
            if seen & set(b): raise ValueError(f"overlapping segments: {sorted(seen & set(b))}")
            seen |= set(b)
        s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source)
                                 .register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

def win_plus(model, source, heads_by_block, n_heads, window_tf=IDEN, inner=IDEN, window=None):
    """Rotations off in the window (window_tf), plus `inner` on the listed heads. heads_by_block: {block: [heads]},
    blocks outside the window."""
    segs = [(list(WINDOW if window is None else window), window_tf)] if (window is None or window) else []
    for k in sorted(heads_by_block):
        hs = sorted(set(heads_by_block[k]))
        if hs: segs.append(([k], per_head(hs, inner, n_heads)))
    return RopeComposite(model, source, segs)

In [10]:
# algebra gates (as in W5 / RB-0)
def _rot_il(x):
    x = x.unflatten(-1, (-1, 2)); return torch.stack([-x[..., 1], x[..., 0]], -1).flatten(-2)
def _rot_h(x):
    a, b = x.chunk(2, -1); return torch.cat([-b, a], -1)
def _apply(x, e, half=False):
    s, c = e.chunk(2, -1); return x * c + (_rot_h(x) if half else _rot_il(x)) * s

for lname, rh in [("interleaved", False), ("rotate_half", True)]:
    npl, dim, H_ = 32, 64, 12
    e = torch.randn(196, 2 * dim); x = torch.randn(2, H_, 196, dim)
    gate(f"empty plane mask bit-identical [{lname}]", "algebra", torch.equal(mask_planes(e, [], npl, rh), e))
    gate(f"full plane mask equals identity [{lname}]", "algebra",
         torch.allclose(mask_planes(e, list(range(npl)), npl, rh), identity_like(e)))
    m = mask_planes(e, [3], npl, rh)
    touched = torch.nonzero((m - e).abs().sum(0) > 0).flatten().tolist()
    sc = plane_columns(3, npl, rh); exp = sorted(set(sc + [c + dim for c in sc]))
    gate(f"plane 3 touches only its own columns [{lname}]", "algebra", touched == exp, f"{touched} vs {exp}")
    full = _apply(x, e, rh)
    gate(f"mask NO heads equals unmodified rope [{lname}]", "per-head",
         torch.allclose(_apply(x, per_head([], identity_like, H_)(e), rh), full, atol=1e-6))
    one = _apply(x, per_head([3], identity_like, H_)(e), rh); others = [h for h in range(H_) if h != 3]
    gate(f"mask head 3 leaves others untouched [{lname}]", "per-head", torch.allclose(one[:, others], full[:, others], atol=1e-6))
    gate(f"mask head 3 changes head 3 [{lname}]", "per-head", not torch.allclose(one[:, 3], full[:, 3]))
    # single head off after a one-axis window (the window masks that axis's planes, the head masks all)
    ew = mask_planes(e, list(range(0, npl, 2)), npl, rh)
    gate(f"one-axis window leaves the other axis's planes bit-identical [{lname}]", "algebra",
         torch.equal(ew.chunk(2, -1)[0][:, plane_columns(1, npl, rh)], e.chunk(2, -1)[0][:, plane_columns(1, npl, rh)]))

class _Stub:
    def __init__(s, n=12): s.blocks = [nn.Identity() for _ in range(n)]
try:
    _wp = win_plus(_Stub(), "timm", {7: [3]}, 12, window_tf=IDEN)
    dj = [b for b, _ in _wp.segs] == [WINDOW, [7]]
except Exception:
    dj = False
gate("win_plus builds the window plus one segment for the single head's block", "composite", dj)
try:
    win_plus(_Stub(), "timm", {5: [0]}, 12); bad = False
except ValueError:
    bad = True
gate("win_plus REJECTS a single head inside the window", "composite", bad)

  [PASS] algebra: empty plane mask bit-identical [interleaved]


  [PASS] algebra: full plane mask equals identity [interleaved]


  [PASS] algebra: plane 3 touches only its own columns [interleaved]: [6, 7, 70, 71] vs [6, 7, 70, 71]


  [PASS] per-head: mask NO heads equals unmodified rope [interleaved]


  [PASS] per-head: mask head 3 leaves others untouched [interleaved]


  [PASS] per-head: mask head 3 changes head 3 [interleaved]


... (7 outputs left out)

  [PASS] algebra: one-axis window leaves the other axis's planes bit-identical [rotate_half]


  [PASS] composite: win_plus builds the window plus one segment for the single head's block


  [PASS] composite: win_plus REJECTS a single head inside the window


True

In [11]:
# RB-3 helpers: from RB-0 / W7
class RopeGlobal:
    def __init__(s, model, tf): s.name, s.mod = find_rope(model); s.tf = tf; s.handles = []
    def __enter__(s):
        if s.mod is None: raise RuntimeError("no get_embed module")
        s.orig = s.mod.get_embed; o, t = s.orig, s.tf
        s.mod.get_embed = (lambda _o=o, _t=t: (lambda *a, **k: _t(_o(*a, **k))))()
        s.handles = [1]; return s
    def __exit__(s, *e): s.mod.get_embed = s.orig; s.handles = []

def shuffled_like(perm):
    def tf(emb):
        if emb.dim() != 2: raise RuntimeError(f"shuffle expects 2-D rope, got {tuple(emb.shape)}")
        return emb[perm]
    return tf

def heads_only(model, source, heads_by_block, n_heads, inner=IDEN):
    """No window: just the listed (block, head) units (RB-0 part C intact checks)."""
    return RopeComposite(model, source, [([k], per_head(sorted(set(hs)), inner, n_heads))
                                         for k, hs in sorted(heads_by_block.items()) if hs])

class PrefixBlock:
    """W7 C3: block patch-query -> prefix-key attention in every block via an additive attn_mask."""
    def __init__(s, model, source, npt, T):
        s.b = get_vit_blocks(model, source); s.source = source; s.npt = npt; s.N = npt + T; s.h = []
    @property
    def handles(s): return s.h
    def __enter__(s):
        N, npt = s.N, s.npt
        def pre(mod, args, kw):
            x = args[0] if args else kw.get("x")
            m = torch.zeros(1, 1, N, N, device=x.device, dtype=x.dtype)
            m[..., npt:, :npt] = float("-inf")
            kw = dict(kw); old = kw.get("attn_mask", None)
            kw["attn_mask"] = m if old is None else old + m
            return args, kw
        for blk in s.b:
            s.h.append(get_block_attention(blk, s.source).register_forward_pre_hook(pre, with_kwargs=True))
        return s
    def __exit__(s, *e):
        for x in s.h: x.remove()
        s.h = []

_g7 = torch.Generator().manual_seed(W7_PERM_SEED)
PERMS = [torch.randperm(196, generator=_g7) for _ in range(2)]          # W7's perms[0], perms[1]
gate("W7 permutation 0 equals RB-0's shuffle permutation (randperm, seed 7)", "algebra",
     torch.equal(PERMS[0], torch.randperm(196, generator=torch.Generator().manual_seed(W7_PERM_SEED))))
gate("the two W7 permutations differ and are permutations of 0..195", "algebra",
     not torch.equal(PERMS[0], PERMS[1]) and all(sorted(p.tolist()) == list(range(196)) for p in PERMS))
try:
    _ho = heads_only(_Stub(), "timm", {7: [3]}, 12); ho_ok = [b for b, _ in _ho.segs] == [[7]]
except Exception:
    ho_ok = False
gate("heads_only builds one segment for the single head's block and no window", "composite", ho_ok)

  [PASS] algebra: W7 permutation 0 equals RB-0's shuffle permutation (randperm, seed 7)


  [PASS] algebra: the two W7 permutations differ and are permutations of 0..195


  [PASS] composite: heads_only builds one segment for the single head's block and no window


True

In [12]:
# image cache: decode once per model, replay through a real DataLoader (RB-0)
class _CachedBatches(torch.utils.data.IterableDataset):
    def __init__(self, batches): super().__init__(); self.batches = batches
    def __iter__(self):
        for b in self.batches: yield b

def cached_loader(cache):
    return torch.utils.data.DataLoader(_CachedBatches(cache), batch_size=None, num_workers=0,
                                       pin_memory=torch.cuda.is_available(), shuffle=False)

def build_cache(dataset, processor, source, n):
    t = time.time()
    dl = prep_data(dataset, processor, source, corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(im.clone(), torch.as_tensor(lb).clone()) for im, lb in dl]
    got = sum(int(lb.numel()) for _, lb in out)
    print(f"    cached {got} images in {time.time() - t:.0f}s")
    return out

def cache_digest(cache):
    h = hashlib.sha1()
    for im, lb in cache[:2]:
        h.update(im[:4].numpy().tobytes()); h.update(lb[:4].numpy().tobytes())
    return h.hexdigest()[:12]

In [13]:
# measurement: RB-0's measure() without the bootstrap / head-norm extras
def final_norm_module(model):
    n = getattr(model, "norm", None)
    if n is not None and not isinstance(n, nn.Identity): return n, "norm"
    f = getattr(model, "fc_norm", None)
    if f is not None and not isinstance(f, nn.Identity): return f, "fc_norm"
    return nn.Identity(), "identity"

def measure(M, ctx, capture=()):
    """One RPI sweep over the cached images under `ctx`: SSDC at every block input and at FINAL, plus captured patch
    tokens."""
    model, source, n_prefix = M["model"], M["source"], M["n_prefix"]
    cache = M["cache"]["ssdc"]
    blocks = get_vit_blocks(model, source)
    acc = SSDCAccumulator(n_prefix=n_prefix, spatial_metric=METRIC)
    handles = _register_ssdc_accumulator_hooks(model, source, acc)        # the repo's own readout
    fnorm = M["final_norm"]
    st = {"next": 0, "cur": 0}
    cap = {r: [] for r in capture}
    def keep(r, tok):
        if r in cap: cap[r].append(tok[:, n_prefix:, :].detach().float().cpu())
    extra = [model.register_forward_pre_hook(
        lambda m, args: st.update(cur=st["next"], next=st["next"] + int(args[0].shape[0])))]
    for i in range(N_BLOCKS):
        if i in cap:
            extra.append(get_block_attention(blocks[i], source).register_forward_hook(
                lambda m, inp, out, i=i: keep(i, inp[0].detach())))
    def final_hook(m, inp, out):
        t = fnorm(out.detach()); acc.add(FINAL, t); keep(FINAL, t)
    extra.append(blocks[-1].register_forward_hook(final_hook))
    t0 = time.time()
    torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            predict(model, cached_loader(cache), source, RPI=True, half=False)
    finally:
        for h in handles + extra: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(N_BLOCKS)] + [acc.ssdc(FINAL, remove_prefix=True)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": int(st["next"]),
           "runtime_s": round(time.time() - t0, 1)}
    if capture:
        res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    return res

# probes: W5's per-block-seeded protocol from RB-0
def _fit(Xtr, ytr, Xte, yte, nc, seed):
    torch.manual_seed(seed)
    head = nn.Linear(Xtr.shape[1], nc).to(Xtr.device)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR)
    n = Xtr.shape[0]; peak = 0.0; best = None
    for _ in range(PROBE_PASSES):
        perm = torch.randperm(n, device=Xtr.device); head.train()
        for s0 in range(0, n, PROBE_BATCH):
            b = perm[s0:s0 + PROBE_BATCH]
            opt.zero_grad(); F.cross_entropy(head(Xtr[b]), ytr[b]).backward(); opt.step()
        head.eval(); correct = 0; preds = []
        with torch.no_grad():
            for s0 in range(0, Xte.shape[0], PROBE_CHUNK):
                p = head(Xte[s0:s0 + PROBE_CHUNK]).argmax(-1); preds.append(p)
                correct += (p == yte[s0:s0 + PROBE_CHUNK]).sum().item()
        acc_ = correct / Xte.shape[0]
        if acc_ > peak: peak = acc_; best = torch.cat(preds)
    del head, opt
    return float(peak), best

def _split(n_images, T, seed=TORCH_SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(n_images, generator=g); n_tr = int(round(n_images * PROBE_TRAIN_FRAC))
    rows = lambda ids: (ids.unsqueeze(1) * T + torch.arange(T).unsqueeze(0)).reshape(-1)
    return rows(idx[:n_tr]), rows(idx[n_tr:]), idx[n_tr:]

def probe_block(acts, H, W, block, heads=("exact", "row", "col"), noise=False):
    n_images, T, D = acts.shape
    feats = acts.reshape(-1, D)
    tr, te, te_img = _split(n_images, T)
    Xtr = feats[tr].to(PDEV); Xte = feats[te].to(PDEV)
    mu = Xtr.mean(0, keepdim=True); sd = Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    slot = torch.arange(T); out = {}
    labels = {"exact": (slot, T), "row": (slot // W, H), "col": (slot % W, W)}
    for name in heads:
        lab, nc = labels[name]; y = lab.repeat(n_images)
        acc_, preds = _fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block)
        out[name] = acc_
        if noise:
            corr = (preds == y[te].to(PDEV)).float()
            g = np.random.default_rng(TORCH_SEED + block)
            n_te = len(te_img); per = corr.reshape(n_te, T).mean(1).cpu().numpy()
            bt = [per[g.integers(0, n_te, n_te)].mean() for _ in range(NOISE_BOOT)]
            seeds = [_fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block + 1000 * s)[0]
                     for s in NOISE_SEEDS]
            out.setdefault("noise", {})[name] = {
                "boot_sd": float(np.std(bt, ddof=1)), "seed_sd": float(np.std(seeds, ddof=1)),
                "seed_values": [float(v) for v in seeds],
                "combined_sd": float(np.sqrt(np.var(bt, ddof=1) + np.var(seeds, ddof=1)))}
    out["chance"] = {"exact": 1.0 / T, "row": 1.0 / H, "col": 1.0 / W}
    del Xtr, Xte
    if PDEV == "cuda": torch.cuda.empty_cache()
    return out

In [14]:
# top-1 on the cached ILSVRC images: RB-0's accuracy()
def accuracy(M, ctx):
    model, cache = M["model"], M["cache"]["acc"]
    model.eval(); correct = []
    torch.manual_seed(TORCH_SEED)
    with ctx:
        hooks = len(getattr(ctx, "handles", []) or [])
        with torch.inference_mode():
            for images, labels in cached_loader(cache):
                pred = model(images.to(DEVICE)).argmax(-1).cpu()
                correct.append(pred == torch.as_tensor(labels).cpu())
    c = torch.cat(correct)
    return float(c.float().mean()), c.numpy().astype(np.uint8), hooks

In [15]:
# per-model setup (RB-2's, plus the ILSVRC cache and RB-0's top-1 replay gate)
TEST_INIT_SEED = {"naver": 101, "DINOv3": 102, "small": 103, "reg1_gap": 104}
def load_model(key):
    spec = MODEL_SPECS[key]
    if TEST_MODE:
        tiny = dict(embed_dim=96, num_heads=6 if key != "naver" else 12, pretrained=False, dynamic_img_size=True)
        if key == "naver": tiny["embed_dim"] = 192
        tiny["num_classes"] = 10 if spec["classifier"] else 0
        torch.manual_seed(TEST_INIT_SEED[key])
        model = timm.create_model(spec["name"].split(".")[0], **tiny).to(DEVICE).eval()
        with torch.no_grad():
            for blk in model.blocks:
                for gname in ("gamma_1", "gamma_2"):
                    g_ = getattr(blk, gname, None)
                    if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                q_ = getattr(blk.attn, "qkv", None)
                if q_ is not None: q_.weight.mul_(4.0)
                for pn in ("q_proj", "k_proj"):
                    m_ = getattr(blk.attn, pn, None)
                    if m_ is not None: m_.weight.mul_(4.0)
        cfg = timm.data.resolve_model_data_config(model); cfg["input_size"] = (3, INPUT_SIZE, INPUT_SIZE)
        return model, timm.data.create_transform(**cfg, is_training=False), "timm"
    return load_rope(model_name=spec["name"], device=DEVICE, half=False, input_size=INPUT_SIZE)

def first_layer_rel_diff(M):
    model, source = M["model"], M["source"]
    conv = get_patch_embed_conv(model, source); grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    try:
        with torch.no_grad(): model(M["cache"]["ssdc"][0][0].to(DEVICE))
    finally:
        h.remove()
    with torch.no_grad():
        y64 = F.conv2d(grab["x"].double(), conv.weight.double(), None if conv.bias is None else conv.bias.double(),
                       conv.stride, conv.padding, conv.dilation, conv.groups)
        v = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
    n = int(grab["x"].shape[0]); del grab, y64
    return v, n

def setup_model(key):
    spec = MODEL_SPECS[key]
    print("\n" + "=" * 78 + f"\n{key}  ({spec['name']})\n" + "=" * 78, flush=True)
    model, processor, source = load_model(key)
    model.eval().float()
    rh, rh_src, nh_attr = attn_attrs(model, source)
    n_prefix = int(getattr(model, "num_prefix_tokens", 1))
    M = {"key": key, "model": model, "source": source, "processor": processor, "n_prefix": n_prefix,
         "rotate_half": bool(rh), "n_heads": int(nh_attr), "cache": {}}
    M["cache"]["ssdc"] = build_cache(DS_SSDC, processor, source, N_SSDC)
    if spec["classifier"]:
        M["cache"]["acc"] = build_cache(DS_ACC, processor, source, N_ACC)
    with torch.no_grad():
        g = get_patch_embed_conv(model, source)(M["cache"]["ssdc"][0][0][:1].to(DEVICE))
    Hg, Wg = int(g.shape[-2]), int(g.shape[-1]); M["grid"] = (Hg, Wg); T = Hg * Wg
    gate("forward grid is 14x14", key, (Hg, Wg) == (14, 14), f"{Hg}x{Wg}")
    fn, fn_name = final_norm_module(model); M["final_norm"] = fn; M["final_norm_name"] = fn_name
    A_ = ANCH.get(key)
    if A_:
        gate("num_heads matches W5", key, M["n_heads"] == A_["n_heads"], f"{M['n_heads']} vs {A_['n_heads']}")
    ra = (RB0.get(key) or {}).get("A")
    if ra:
        gate("num_heads matches RB-0", key, M["n_heads"] == ra["n_heads"], f"{M['n_heads']} vs {ra['n_heads']}")
    rname, rmod = find_rope(model); M["rope"] = rmod
    emb0 = rmod.get_embed(shape=(Hg, Wg))
    n_planes = int(emb0.shape[-1] // 4)
    amap, rowp, colp, dec = derive_axis(emb0, n_planes, rh, Hg, Wg)
    gate("rope rows equal patch tokens (196, as the W7 permutations assume)", key, int(emb0.shape[0]) == T == 196, f"{emb0.shape[0]} vs {T}")
    gate("rotary table has no head dimension", key, emb0.dim() == 2, f"shape {tuple(emb0.shape)}")
    if A_:
        gate("row/col planes equal W5's runtime derivation", key, rowp == A_["row_planes"] and colp == A_["col_planes"], f"row {rowp[:3]}..")
    if DEVICE == "cuda" and not TEST_MODE:
        v, nb = first_layer_rel_diff(M); exp_ = FIRST_LAYER_910[key]; M["first_layer_rel_diff"] = v
        gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value (TF32-level kernel, as in September)", key,
             nb == BATCH_SIZE and abs(v / exp_ - 1) <= 0.01, f"{v:.4e} vs {exp_:.4e} (bit-identical: {v == exp_}; batch {nb})")
    else:
        gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value", key, True, "test mode / CPU", skip=True)
    def streamed_ssdc(ctx):
        torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
        with ctx:
            s, _ = evaluate_ssdc(model, processor, DS_SSDC, source, RPI=True, number_images=N_SSDC,
                                 batch_size=BATCH_SIZE, metric=METRIC, half=False, num_workers=0)
        return s
    for nm_, mk in [("baseline", NullCtx), ("Cref", lambda: RopeComposite(model, source, [(WINDOW, IDEN)]))]:
        a = streamed_ssdc(mk()); b = measure(M, mk())["ssdc"][:N_BLOCKS]
        d = max(abs(x - y) for x, y in zip(a, b))
        gate(f"cached replay reproduces streamed evaluate_ssdc [{nm_}]", key, d <= EXACT_TOL, f"max |delta| {d:.2e}")
    if "acc" in M["cache"]:
        n_chk = min(REPLAY_CHECK_N, N_ACC)
        dl = prep_data(DS_ACC, processor, source, corruption_type=None, number_images=n_chk,
                       batch_size=BATCH_SIZE, half=False, num_workers=0)
        cs = []
        with torch.inference_mode():
            for im, lb in dl:
                cs.append(model(im.to(DEVICE)).argmax(-1).cpu() == torch.as_tensor(lb))
        cs = torch.cat(cs).numpy().astype(np.uint8)
        _, cc, _ = accuracy(M, NullCtx())
        gate(f"cached replay reproduces streamed top-1 on the first {n_chk} images", key,
             np.array_equal(cs, cc[:n_chk]), f"{int((cs != cc[:n_chk]).sum())} images differ")
    return M

In [16]:
# bookkeeping: one file per model, saved after every arm group
RESULTS = {}
CFG_FP = {"n_ssdc": N_SSDC, "n_acc": N_ACC, "batch": BATCH_SIZE, "seed": TORCH_SEED, "ssdc_dataset": SSDC_DATASET,
          "acc_dataset": ACC_DATASET, "metric": METRIC, "window": WINDOW, "post": POST, "top_shuffle": TOP_SHUFFLE,
          "perm_seed": W7_PERM_SEED, "n_rand_gate": N_RAND_GATE, "test_mode": TEST_MODE}
SESSION = {"session": RUN_STAMP, "torch": torch.__version__, "cudnn": ENV["cudnn_reported"], "python": sys.executable,
           "gpu": ENV["gpu"], "repo_commit": REPO_COMMIT}
def model_file(key): return OUT_DIR / f"rb3_{key}.json"
def img_file(key): return OUT_DIR / f"rb3_perimage_{key}.npz"

def load_R(key):
    f = model_file(key)
    if not (RESUME and f.exists()): return None, {}
    R = json.loads(f.read_text())
    if R.get("config") != CFG_FP:
        raise SystemExit(f"{f.name} was written with a different configuration; move it away or set RB3_RESUME=0.")
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    for n, r in R["acc"].items():
        if n not in P: raise SystemExit(f"{img_file(key).name} lacks per-image results for {n}; set RB3_RESUME=0.")
    return R, P

def save_R(key, R, P):
    R["saved_utc"] = datetime.now(timezone.utc).isoformat()
    if P:
        tmpi = img_file(key).with_name(img_file(key).stem + ".tmp.npz")
        np.savez_compressed(tmpi, **P); os.replace(tmpi, img_file(key))
    tmp = model_file(key).with_suffix(".json.tmp")
    tmp.write_text(json.dumps(R, indent=1, default=float)); os.replace(tmp, model_file(key))

def plan(key, nh):
    """Every arm RB-3 measures for this model: name -> (kind, builder spec). Fixed before the run."""
    arms = {}
    if MODEL_SPECS[key]["classifier"]:
        arms["baseline"] = ("acc", ("null",)); arms["window"] = ("acc", ("window",)); arms["floor"] = ("acc", ("floor",))
        for k in POST:
            for h in range(nh): arms[f"int_s_b{k}_h{h}"] = ("acc", ("int", k, h, None))
        for k, h in SEL[key]["top"][:3]: arms[f"s_b{k}_h{h}"] = ("acc", ("win", k, h, None))     # = RB-0 C (gate)
        for k, h in SEL[key]["top"]:
            for p in (0, 1):
                arms[f"s_b{k}_h{h}_shuf{p}"] = ("acc", ("win", k, h, p))
                arms[f"int_s_b{k}_h{h}_shuf{p}"] = ("acc", ("int", k, h, p))
        rb0c = (RB0[key]["C"] or {}).get("conditions", {})
        for n in rb0c:                                               # RB-0's own shuffled single (old rule): gate
            if n.startswith("s_b") and n.endswith("_shuffled"):
                k, h = int(n.split("_")[1][1:]), int(n.split("_")[2][1:])
                arms[f"s_b{k}_h{h}_shuf0"] = ("acc", ("win", k, h, 0))
        r1 = (W7.get(key) or {}).get("top", 0) if not TEST_MODE else 0
        arms["floor_shuffled_s0"] = ("acc", ("floor_shuf", 0)); arms["floor_shuffled_s1"] = ("acc", ("floor_shuf", 1))
        arms["window_shuffled"] = ("acc", ("window_shuf", 0))
        arms[f"win_h{r1}_shuffled"] = ("acc", ("grp_win", r1, 0)); arms[f"int_h{r1}_shuffled"] = ("acc", ("grp_int", r1, 0))
        arms["prefix_block"] = ("acc", ("prefix",))
    arms["ssdc_Cref"] = ("ssdc", ("window",))
    sel = SEL[key]
    arms["ssdc_best_per_block"] = ("ssdc", ("set", {k: v for k, v in sel["best_per_block"].items()}))
    arms["ssdc_top6"] = ("ssdc", ("set", sel["top6"]))
    rand = (RB0[key]["D"] or {}).get("random_sets", [])
    for i, s in enumerate(rand[:N_RAND_GATE]):
        arms[f"ssdc_rand{i:02d}"] = ("ssdc", ("set", {k: [h] for k, h in zip(POST, s)}))
    if MODEL_SPECS[key]["classifier"]:
        arms["best_per_block"] = ("acc", ("set", sel["best_per_block"])); arms["top6"] = ("acc", ("set", sel["top6"]))
        for i, s in enumerate(rand[:N_RAND_GATE]):
            arms[f"rand{i:02d}"] = ("acc", ("set", {k: [h] for k, h in zip(POST, s)}))
    return arms

def build_ctx(M, spec):
    model, source, nh = M["model"], M["source"], M["n_heads"]
    kind = spec[0]
    if kind == "null": return NullCtx()
    if kind == "window": return RopeComposite(model, source, [(WINDOW, IDEN)])
    if kind == "floor": return RopeGlobal(model, IDEN)
    if kind == "floor_shuf": return RopeGlobal(model, shuffled_like(PERMS[spec[1]]))
    if kind == "window_shuf": return RopeComposite(model, source, [(WINDOW, shuffled_like(PERMS[spec[1]]))])
    if kind in ("win", "int"):
        _, k, h, p = spec; inner = IDEN if p is None else shuffled_like(PERMS[p])
        return (win_plus(model, source, {k: [h]}, nh, inner=inner) if kind == "win" else heads_only(model, source, {k: [h]}, nh, inner=inner))
    if kind == "grp_win":   # W7's construction: window, then one post segment with the head index masked
        return RopeComposite(model, source, [(WINDOW, IDEN), (POST, per_head([spec[1]], shuffled_like(PERMS[spec[2]]), nh))])
    if kind == "grp_int":
        return RopeComposite(model, source, [(POST, per_head([spec[1]], shuffled_like(PERMS[spec[2]]), nh))])
    if kind == "prefix": return PrefixBlock(model, source, M["n_prefix"], 196)
    if kind == "set": return win_plus(model, source, {int(k): v for k, v in spec[1].items()}, nh)
    raise ValueError(spec)

def expected_hooks(spec, nh):
    kind = spec[0]
    return {"null": 0, "window": len(WINDOW), "floor": 1, "floor_shuf": 1, "window_shuf": len(WINDOW), "win": len(WINDOW) + 1,
            "int": 1, "grp_win": len(WINDOW) + len(POST), "grp_int": len(POST), "prefix": N_BLOCKS,
            "set": len(WINDOW) + sum(1 for v in (spec[1].values() if kind == "set" else []) if v)}[kind]

## Gates on the saved results
All read the saved file, so a resumed or re-loaded run still checks every arm. "Bit for bit" = identical values.

In [17]:
def _maxdiff(a, b): return max((abs(x - y) for x, y in zip(a, b)), default=0.0)

def check_refs(key, R, P):
    ok = []; rb = RB0[key]
    if MODEL_SPECS[key]["classifier"]:
        cc, Pc = rb["C"]["conditions"], rb["P"]
        bad = [n for n in ("baseline", "window", "floor") if R["acc"][n]["acc"] != cc[n]["acc"] or not np.array_equal(P[n], Pc[n])]
        ok.append(gate("top-1 references (baseline, window, floor) equal RB-0 part C bit for bit (mean and per image)", key, not bad,
                       ", ".join(f"{n} {R['acc'][n]['acc'] * 100:.2f}" for n in ("baseline", "window", "floor"))))
        if key in W7:
            w7, wi = W7[key]["acc"], W7[key]["img"]
            pairs = [("baseline", "baseline"), ("window", "window_identity"), ("floor", "floor_identity")]
            dev = max(abs(R["acc"][a]["acc"] - w7[b]) for a, b in pairs)
            if W7_EXACT[key]:
                ok.append(gate("top-1 references equal W7 bit for bit (cuDNN-insensitive model)", key, dev == 0.0, f"max |delta| {dev * 100:.2f} pts"))
            else:
                gate("top-1 references vs W7 recorded (ViT-S: W7 used cuDNN 9.26-like numerics)", key, True,
                     f"max |delta| {dev * 100:.2f} pts; per-image agreement "
                     + ", ".join(f"{a} {float((P[a] == wi[b]).mean()) * 100:.2f}%" for a, b in pairs if b in wi))
    d = _maxdiff(R["ssdc"]["ssdc_Cref"]["ssdc"], rb["A"]["conditions"]["Cref"]["ssdc"])
    ok.append(gate("Cref SSDC equals RB-0 part A bit for bit (13 readouts)", key, d <= EXACT_TOL, f"max |delta| {d:.1e}"))
    return all(ok)

def check_arms(key, R, P, nh):
    arms = plan(key, nh); rb = RB0[key]
    missing = [n for n, (kind, _) in arms.items() if n not in R[kind]]
    gate("every planned arm measured", key, not missing, f"{len(arms) - len(missing)}/{len(arms)}" + (f"; missing {missing[:4]}" if missing else ""))
    badh = [n for n, (kind, spec) in arms.items() if n in R[kind] and R[kind][n]["hooks"] != expected_hooks(spec, nh)]
    gate("every arm registered exactly the expected hooks", key, not badh, ",".join(badh[:5]))
    badn = [n for n, (kind, _) in arms.items() if n in R[kind] and R[kind][n]["n_images"] != (N_ACC if kind == "acc" else N_SSDC)]
    gate("every arm saw every image", key, not badn, ",".join(badn[:5]))
    if MODEL_SPECS[key]["classifier"]:
        cc, Pc = rb["C"]["conditions"], rb["P"]
        # RB-0 arms re-measured: intact / window singles, shuffled single, index-group shuffled arm
        same = {}
        for n in R["acc"]:
            src = n
            if n.endswith("_shuf0") and n.startswith("s_b"): src = n[:-len("_shuf0")] + "_shuffled"
            if src in cc and n not in ("baseline", "window", "floor"):
                same[n] = src
        bad = [n for n, s in same.items() if R["acc"][n]["acc"] != cc[s]["acc"] or not np.array_equal(P[n], Pc[s])]
        gate("every arm RB-0 part C also ran equals it bit for bit (mean and per image)", key, bool(same) and not bad,
             f"{len(same)} arms: {sorted(same)[:6]}{'...' if len(same) > 6 else ''}" + (f"; differ {bad[:4]}" if bad else ""))
        D = rb["D"]
        badr = [f"rand{i:02d}" for i in range(N_RAND_GATE) if f"rand{i:02d}" in R["acc"] and
                (R["acc"][f"rand{i:02d}"]["acc"] != D["acc"][f"rand{i:02d}"] or not np.array_equal(P[f"rand{i:02d}"], Pc[f"D_rand{i:02d}"]))]
        gate(f"RB-0 part D random sets 0-{N_RAND_GATE - 1} reproduce bit for bit (top-1, mean and per image)", key, not badr, ",".join(badr))
        if key in W7:
            w7, wi = W7[key]["acc"], W7[key]["img"]; r1 = W7[key]["top"]
            pairs = [(n, n) for n in ("floor_shuffled_s0", "floor_shuffled_s1", "window_shuffled", f"win_h{r1}_shuffled", f"int_h{r1}_shuffled")]
            dev = max(abs(R["acc"][a]["acc"] - w7[b]) for a, b in pairs)
            agree = {a: float((P[a] == wi[b]).mean()) for a, b in pairs if b in wi}
            dpb = abs(R["acc"]["prefix_block"]["acc"] - W7[key]["prefix_block_acc"])
            if W7_EXACT[key]:
                gate("W7 Table 18 shuffled arms reproduce W7 bit for bit (cuDNN-insensitive model)", key,
                     dev == 0.0 and all(v == 1.0 for v in agree.values()), f"max |delta| {dev * 100:.2f} pts; per-image {len(agree)} arms all equal: {all(v == 1.0 for v in agree.values())}")
                gate("prefix-blocked top-1 (Table 10) reproduces W7 C3 bit for bit", key, dpb == 0.0,
                     f"{R['acc']['prefix_block']['acc'] * 100:.2f} vs {W7[key]['prefix_block_acc'] * 100:.2f}")
            else:
                gate("W7 Table 18 / Table 10 arms vs W7 recorded (ViT-S: 9.10.2 values replace W7's)", key, True,
                     f"max |delta| {dev * 100:.2f} pts; prefix-blocked {dpb * 100:.2f} pts; per-image agreement "
                     + ", ".join(f"{a} {v * 100:.2f}%" for a, v in agree.items()))
    D = rb["D"]
    badr = [f"rand{i:02d}" for i in range(N_RAND_GATE) if f"ssdc_rand{i:02d}" in R["ssdc"] and
            _maxdiff(R["ssdc"][f"ssdc_rand{i:02d}"]["ssdc"], D["ssdc"][f"rand{i:02d}"]) > EXACT_TOL]
    gate(f"RB-0 part D random sets 0-{N_RAND_GATE - 1} reproduce bit for bit (SSDC, 13 readouts)", key, not badr, ",".join(badr))
    if not TEST_MODE:
        ses = R.get("sessions", [])
        gate("every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002", key,
             bool(ses) and all(s.get("cudnn") == EXPECT_CUDNN and s.get("torch") == EXPECT_TORCH for s in ses),
             f"{len(ses)} session(s)")

## Measurement
Per model: the references and their gates first (a failed reference gate stops that model when `RB3_STRICT=1`),
then every planned arm. Saves after every 12 arms; a re-run continues where it stopped.

In [18]:
def run_model(M):
    key, nh = M["key"], M["n_heads"]
    R, P = load_R(key)
    if R is None:
        R, P = {"model": key, "name": MODEL_SPECS[key]["name"], "notebook": "RB-3", "config": CFG_FP, "n_heads": nh,
                "selection": {"rule": RULE_RANK, "top": SEL[key]["top"], "best_per_block": {str(k): v for k, v in SEL[key]["best_per_block"].items()},
                              "top6": {str(k): v for k, v in SEL[key]["top6"].items()}},
                "acc": {}, "ssdc": {}, "sessions": []}, {}
    else:
        print(f"  resuming {model_file(key).name}: {len(R['acc'])} top-1 arms, {len(R['ssdc'])} SSDC arms already saved")
    R["sessions"].append({**SESSION, "first_layer_rel_diff": M.get("first_layer_rel_diff")})
    arms = plan(key, nh); t0 = time.time(); n_new = [0]
    def run(name):
        kind, spec = arms[name]
        if name in R[kind]: return
        ctx = build_ctx(M, spec)
        if kind == "acc":
            a, c, hooks = accuracy(M, ctx); R["acc"][name] = {"acc": a, "hooks": hooks, "n_images": int(len(c)), "spec": str(spec)}; P[name] = c
        else:
            r = measure(M, ctx); R["ssdc"][name] = {"ssdc": r["ssdc"], "hooks": r["hooks"], "n_images": r["n_images"], "spec": str(spec)}
        n_new[0] += 1
    refs = [n for n in ("baseline", "window", "floor", "ssdc_Cref") if n in arms]
    for n in refs: run(n)
    save_R(key, R, P)
    if not check_refs(key, R, P) and STRICT and not TEST_MODE:
        gate("arms measured", key, False, "STOPPED: a reference gate failed (RB3_STRICT=1). Send the gate report.")
        return R
    todo = [n for n in arms if n not in refs and n not in R[arms[n][0]]]
    print(f"\n--- {key}: {len(todo)} arms to measure ---", flush=True)
    for i, n in enumerate(todo):
        run(n)
        if (i + 1) % 12 == 0 or i == len(todo) - 1:
            save_R(key, R, P); el = time.time() - t0
            print(f"    {i + 1}/{len(todo)} arms | {el / 60:.1f} min, ~{el / (i + 1) * (len(todo) - i - 1) / 60:.0f} min left", flush=True)
    R["complete"] = all(n in R[arms[n][0]] for n in arms)
    save_R(key, R, P)
    check_arms(key, R, P, nh)
    gate("model finished", key, True, f"{n_new[0]} new arms, {(time.time() - t0) / 60:.1f} min; complete {R['complete']}")
    return R

In [19]:
T_RUN = time.time()
for key in MODELS_RUN:
    if key not in SEL:
        gate("model measured", key, False, "no RB-0 part A selection"); continue
    R0, P0 = load_R(key)
    if R0 and R0.get("complete"):
        print(f"\n{key}: already complete in {model_file(key).name}; loading and re-checking")
        RESULTS[key] = R0; check_refs(key, R0, P0); check_arms(key, R0, P0, R0["n_heads"]); continue
    try:
        M = setup_model(key)
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", key, False, repr(ex)[:200]); continue
    setup_fail = [g for g in GATES if g["scope"] == key and g["status"] == "FAIL"]
    if setup_fail and STRICT and not TEST_MODE:
        gate("model measured", key, False, f"STOPPED: {len(setup_fail)} setup gate(s) failed: " + "; ".join(g["gate"] for g in setup_fail)[:300])
        del M; continue
    try:
        RESULTS[key] = run_model(M)
    except Exception as ex:
        traceback.print_exc(); gate("model measured", key, False, repr(ex)[:200])
    del M
    if DEVICE == "cuda": torch.cuda.empty_cache()
print(f"\nmeasurement finished in {(time.time() - T_RUN) / 60:.1f} min")


naver  (vit_base_patch16_rope_224.naver_in1k)


    cached 1000 images in 6s


    cached 5000 images in 41s
  [PASS] naver: forward grid is 14x14: 14x14


  [PASS] naver: num_heads matches W5: 12 vs 12


  [PASS] naver: num_heads matches RB-0: 12 vs 12


  [PASS] naver: rope rows equal patch tokens (196, as the W7 permutations assume): 196 vs 196


... (111 outputs left out)

  [PASS] reg1_gap: RB-0 part D random sets 0-1 reproduce bit for bit (SSDC, 13 readouts)


  [PASS] reg1_gap: every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002: 1 session(s)


  [PASS] reg1_gap: model finished: 109 new arms, 8.2 min; complete True



measurement finished in 22.8 min


## Analysis
Reads only saved files. Top-1 costs are points lost against the matching reference with paired per-image SEs:
window arms against the 0-5 window, intact arms against the unmodified model. Window costs of every single head
come from RB-0 part C (the same images and code; the overlap is gated above).

In [20]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

def paired(a, b):
    d = a.astype(float) - b.astype(float); return float(d.mean() * 100), float(d.std(ddof=1) / np.sqrt(len(d)) * 100)

def show(title, obj):
    print("\n" + title); print("-" * len(title))
    if isinstance(obj, pd.DataFrame): print(obj.to_string(index=False) if len(obj) else "(empty)")
    else: print(json.dumps(obj, indent=1, default=float))

ANALYSIS, TABLES = {}, {k: [] for k in ("singles", "table18_index_groups", "shuffled", "null")}
for key in MODELS_RUN:
    R = RESULTS.get(key) or (json.loads(model_file(key).read_text()) if model_file(key).exists() else None)
    if not R: print(f"{key}: nothing saved, skipped"); continue
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    rb = RB0[key]; nh = R["n_heads"]; sel = SEL[key]; out = {"top_by_rule": sel["top"]}
    cost_ssdc = sel["cost"]
    if MODEL_SPECS[key]["classifier"] and all(n in R["acc"] for n in ("baseline", "window")):
        Pc = rb["P"]; cc = rb["C"]["conditions"]
        # single heads: window (RB-0) and intact (RB-3)
        rows = []
        for k in POST:
            for h in range(nh):
                s, i = f"s_b{k}_h{h}", f"int_s_b{k}_h{h}"
                row = {"model": key, "block": k, "head": h, "ssdc_cost": cost_ssdc[(k, h)], "ssdc_readout": "b11" if k < 11 else "out11"}
                if s in Pc: row["window_cost"], row["window_se"] = paired(Pc["window"], Pc[s])
                if i in P: row["intact_cost"], row["intact_se"] = paired(P["baseline"], P[i])
                if s in Pc and i in P:
                    d = (Pc["window"].astype(float) - Pc[s]) - (P["baseline"].astype(float) - P[i])
                    row["window_minus_intact"], row["window_minus_intact_se"] = float(d.mean() * 100), float(d.std(ddof=1) / np.sqrt(len(d)) * 100)
                rows.append(row)
        df = pd.DataFrame(rows); TABLES["singles"] += rows
        d610 = df[df.block < 11]
        top = sel["top"][0]; tk = f"s_b{top[0]}_h{top[1]}"
        others = [f"s_b{k}_h{h}" for k in POST for h in range(nh) if (k, h) != tuple(top)]
        oth_mean = np.mean([Pc["window"].astype(float) - Pc[o] for o in others], 0)
        dtop = (Pc["window"].astype(float) - Pc[tk]) - oth_mean
        by_win = df.sort_values("window_cost", ascending=False).reset_index(drop=True)
        rank_top = int(by_win.index[(by_win.block == top[0]) & (by_win["head"] == top[1])][0]) + 1
        cst = by_win.iloc[0]; nxt = by_win.iloc[1] if rank_top == 1 else by_win.iloc[0]
        nk = f"s_b{int(nxt.block)}_h{int(nxt['head'])}"
        out["claim_window"] = {
            "rule_top": list(top), "rule_top_window_cost": [float(df[(df.block == top[0]) & (df["head"] == top[1])].window_cost.iloc[0]),
                                                           float(df[(df.block == top[0]) & (df["head"] == top[1])].window_se.iloc[0])],
            "rule_top_rank_of_72": rank_top, "n_heads_total": int(len(df)),
            "costliest": [int(cst.block), int(cst["head"]), float(cst.window_cost), float(cst.window_se)],
            "costliest_ssdc_rank_6_10": (sel["rank_6_10"].index((int(cst.block), int(cst["head"]))) + 1) if int(cst.block) < 11 else None,
            "rule_top_minus_" + ("next" if rank_top == 1 else "costliest"): list(paired(Pc[nk], Pc[tk])) if nk != tk else [0.0, 0.0],
            "rule_top_minus_mean_of_others": [float(dtop.mean() * 100), float(dtop.std(ddof=1) / np.sqrt(len(dtop)) * 100)],
            "chance_rule_top_is_costliest": 1.0 / len(df),
        }
        # intact model
        if "intact_cost" in df:
            mx = df.loc[df.intact_cost.idxmax()]
            out["claim_intact"] = {
                "max_intact_cost_any_head": [int(mx.block), int(mx["head"]), float(mx.intact_cost), float(mx.intact_se)],
                "n_heads_intact_cost_above_3se": int(((df.intact_cost - 3 * df.intact_se) > 0).sum()),
                "top_heads": [{"head": [k, h], **{c_: float(df[(df.block == k) & (df["head"] == h)][c_].iloc[0])
                               for c_ in ("window_cost", "window_se", "intact_cost", "intact_se", "window_minus_intact", "window_minus_intact_se")}}
                              for k, h in sel["top"]],
                "spearman_window_vs_intact": float(pd.Series(df.window_cost).corr(pd.Series(df.intact_cost), method="spearman")),
            }
        # shuffled rotations
        sh = []
        for k, h in sel["top"]:
            rec = {"model": key, "block": k, "head": h}
            for p in (0, 1):
                a, b = f"s_b{k}_h{h}_shuf{p}", f"int_s_b{k}_h{h}_shuf{p}"
                if a in P: rec[f"window_shuffled{p}_cost"], rec[f"window_shuffled{p}_se"] = paired(Pc["window"] if "window" not in P else P["window"], P[a])
                if b in P: rec[f"intact_shuffled{p}_cost"], rec[f"intact_shuffled{p}_se"] = paired(P["baseline"], P[b])
            s = f"s_b{k}_h{h}"
            if s in Pc: rec["window_identity_cost"], rec["window_identity_se"] = paired(Pc["window"], Pc[s])
            if f"int_s_b{k}_h{h}" in P: rec["intact_identity_cost"], rec["intact_identity_se"] = paired(P["baseline"], P[f"int_s_b{k}_h{h}"])
            if f"s_b{k}_h{h}_shuf0" in P and s in Pc:
                rec["shuffled0_minus_identity_window"] = list(paired(Pc[s], P[f"s_b{k}_h{h}_shuf0"]))
            sh.append(rec)
        out["shuffled"] = sh; TABLES["shuffled"] += sh
        # Table 18 index groups, cuDNN 9.10.2 (RB-0 C + RB-3 C)
        r1 = (W7.get(key) or {}).get("top", 0) if not TEST_MODE else 0
        t18 = []
        for h in range(nh):
            rec = {"model": key, "head": h, "window_cost": paired(Pc["window"], Pc[f"win_h{h}"])[0], "window_se": paired(Pc["window"], Pc[f"win_h{h}"])[1],
                   "intact_cost": paired(Pc["baseline"], Pc[f"int_h{h}"])[0], "intact_se": paired(Pc["baseline"], Pc[f"int_h{h}"])[1]}
            if key in W7:
                w7 = W7[key]["acc"]; rec["w7_window_cost"] = (w7["window_identity"] - w7[f"win_h{h}"]) * 100
                rec["w7_intact_cost"] = (w7["baseline"] - w7[f"int_h{h}"]) * 100
            t18.append(rec)
        ref = {"intact_top1": R["acc"]["baseline"]["acc"] * 100, "window_top1": R["acc"]["window"]["acc"] * 100,
               "window_shuffled_top1": R["acc"].get("window_shuffled", {}).get("acc", float("nan")) * 100,
               "floor_top1": R["acc"]["floor"]["acc"] * 100,
               "floor_shuffled_top1": [R["acc"].get(f"floor_shuffled_s{p}", {}).get("acc", float("nan")) * 100 for p in (0, 1)],
               "top_group_shuffled_window_cost": (paired(Pc["window"], P[f"win_h{r1}_shuffled"]) if f"win_h{r1}_shuffled" in P else None),
               "top_group_shuffled_intact_cost": (paired(P["baseline"], P[f"int_h{r1}_shuffled"]) if f"int_h{r1}_shuffled" in P else None),
               "prefix_blocked_top1": R["acc"].get("prefix_block", {}).get("acc", float("nan")) * 100, "top_group": r1}
        if key in W7:
            w7 = W7[key]["acc"]
            ref["w7"] = {"intact": w7["baseline"] * 100, "window": w7["window_identity"] * 100, "window_shuffled": w7["window_shuffled"] * 100,
                         "floor": w7["floor_identity"] * 100, "floor_shuffled": [w7["floor_shuffled_s0"] * 100, w7["floor_shuffled_s1"] * 100],
                         "top_group_shuffled_window_cost": (w7["window_identity"] - w7[f"win_h{r1}_shuffled"]) * 100,
                         "top_group_shuffled_intact_cost": (w7["baseline"] - w7[f"int_h{r1}_shuffled"]) * 100,
                         "prefix_blocked_top1": (W7[key]["prefix_block_acc"] or float("nan")) * 100}
        out["table18_index_groups"] = t18; out["table18_references"] = ref; TABLES["table18_index_groups"] += t18
    # index-grouping null (SSDC entering block 11; top-1) with the pre-registered rule
    A = rb["A"]["conditions"]; D = rb["D"]; cref = A["Cref"]["ssdc"]
    rand_s = np.array([cref[11] - D["ssdc"][f"rand{i:02d}"][11] for i in range(len(D["random_sets"]))])
    grp = {h: cref[11] - A[f"grp_h{h}"]["ssdc"][11] for h in range(nh)}
    paper_top = (ANCH.get(key) or {}).get("stageC", [max(grp, key=grp.get)])[0]
    nul = {"model": key, "metric": "ssdc_b11", "n_random": int(len(rand_s)), "random_mean": float(rand_s.mean()),
           "random_p95": float(np.percentile(rand_s, 95)), "random_max": float(rand_s.max()),
           "paper_top_group": int(paper_top), "paper_top_group_cost": float(grp[paper_top]),
           "paper_top_group_percentile": float((rand_s < grp[paper_top]).mean() * 100),
           "n_index_groups_above_random_p95": int(sum(v > np.percentile(rand_s, 95) for v in grp.values()))}
    for n in ("best_per_block", "top6"):
        if f"ssdc_{n}" in R["ssdc"]:
            v = cref[11] - R["ssdc"][f"ssdc_{n}"]["ssdc"][11]; nul[f"{n}_cost"] = float(v); nul[f"{n}_percentile"] = float((rand_s < v).mean() * 100)
    nulls = [nul]
    if MODEL_SPECS[key]["classifier"] and "window" in R["acc"]:
        Pc = rb["P"]; cc = rb["C"]["conditions"]; win = cc["window"]["acc"]
        rand_a = np.array([(win - D["acc"][f"rand{i:02d}"]) * 100 for i in range(len([k_ for k_ in D["acc"] if k_.startswith("rand")]))])
        ga = {h: (win - cc[f"win_h{h}"]["acc"]) * 100 for h in range(nh)}
        na = {"model": key, "metric": "top1", "n_random": int(len(rand_a)), "random_mean": float(rand_a.mean()),
              "random_p95": float(np.percentile(rand_a, 95)), "random_max": float(rand_a.max()),
              "paper_top_group": int(paper_top), "paper_top_group_cost": float(ga[paper_top]),
              "paper_top_group_percentile": float((rand_a < ga[paper_top]).mean() * 100),
              "n_index_groups_above_random_p95": int(sum(v > np.percentile(rand_a, 95) for v in ga.values()))}
        for n in ("best_per_block", "top6"):
            if n in R["acc"]:
                v = (win - R["acc"][n]["acc"]) * 100; na[f"{n}_cost"] = float(v); na[f"{n}_percentile"] = float((rand_a < v).mean() * 100)
        nulls.append(na)
    out["null"] = nulls; TABLES["null"] += nulls
    ANALYSIS[key] = out

for key, out in ANALYSIS.items():
    print("\n" + "=" * 78 + f"\n{key}  ({MODEL_SPECS[key]['label']})   top heads by the rule: {out['top_by_rule']}\n" + "=" * 78)
    if "claim_window" in out: show("window (0-5 off): the rule's top single head vs every other single head (RB-0 C)", out["claim_window"])
    if "claim_intact" in out: show("intact model (RB-3 A)", out["claim_intact"])
    if "shuffled" in out: show("shuffled rotations (RB-3 B), points", pd.DataFrame(out["shuffled"]).drop(columns=["model"], errors="ignore").round(2))
    if "table18_index_groups" in out:
        show("Table 18 index groups, cuDNN 9.10.2 (w7_* = the paper's numbers)", pd.DataFrame(out["table18_index_groups"]).drop(columns=["model"], errors="ignore").round(2))
        show("Table 18 / Table 10 references", out["table18_references"])
    show("index-grouping null (random one-head-per-block sets from RB-0 D)", pd.DataFrame(out["null"]).drop(columns=["model"], errors="ignore").round(4))


naver  (naver ViT-B)   top heads by the rule: [(6, 6), (7, 2), (6, 1), (6, 2)]

window (0-5 off): the rule's top single head vs every other single head (RB-0 C)
--------------------------------------------------------------------------------
{
 "rule_top": [
  6,
  6
 ],
 "rule_top_window_cost": [
  3.04,
  0.28942776937733067
 ],
 "rule_top_rank_of_72": 1,
 "n_heads_total": 72,
 "costliest": [
  6,
  6,
  3.04,
  0.28942776937733067
 ],
 "costliest_ssdc_rank_6_10": 1,
 "rule_top_minus_next": [
  0.12,
  0.26535109240485155
 ],
 "rule_top_minus_mean_of_others": [
  2.8242253521126757,
  0.2597430427230552
 ],
 "chance_rule_top_is_costliest": 0.013888888888888888
}

intact model (RB-3 A)
---------------------
{
 "max_intact_cost_any_head": [
  6,
  3,
  0.4,
  0.19792886390578157
 ],
 "n_heads_intact_cost_above_3se": 0,
 "top_heads": [
  {
   "head": [
    6,
    6
   ],
   "window_cost": 3.04,
   "window_se": 0.28942776937733067,
   "intact_cost": 0.38,
   "intact_se": 0.1427417074264

## Gate report and export

In [21]:
_all = []
for f in sorted(OUT_DIR.glob("rb3_gates_session_*.json")):
    _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT  ({len(set(g['session'] for g in _all))} session(s), {len(_all)} gate calls, latest result per gate shown)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
n_skip = int((gdf.status == "SKIP").sum()) if len(gdf) else 0
complete = {k: bool((RESULTS.get(k) or {}).get("complete")) or (model_file(k).exists() and json.loads(model_file(k).read_text()).get("complete", False)) for k in MODELS_RUN}
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {n_skip} skipped")
print("models complete:", complete)
print("=" * 78)
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: report them before anyone uses the numbers")
for _, r in gdf[gdf.status == "FAIL"].iterrows():
    print(f"   - [{r['scope']}] {r['gate']}: {r['detail']}")
print("=" * 78)

export = {"notebook": "RB-3", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "environment": ENV, "repo": str(repo),
          "repo_commit": REPO_COMMIT, "rb0_dir": str(_rb0) if _rb0 else None, "config": {**CFG_FP, "models": MODELS_RUN, "strict": STRICT},
          "rules": {"rank": RULE_RANK, "best_per_block": RULE_BEST_PER_BLOCK, "top6": RULE_TOP6, "top_shuffle": TOP_SHUFFLE},
          "selection": {k: {"top": v["top"], "best_per_block": {str(a): b for a, b in v["best_per_block"].items()},
                            "top6": {str(a): b for a, b in v["top6"].items()}} for k, v in SEL.items()},
          "analysis": ANALYSIS, "gates": gdf.to_dict("records"),
          "gate_summary": {"total": len(gdf), "fail": n_fail, "skip": n_skip, "complete": complete,
                           "sessions": sorted(set(g["session"] for g in _all))}}
(OUT_DIR / f"rb3_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=float))
gdf.to_csv(OUT_DIR / "rb3_gate_report.csv", index=False)
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"rb3_{name}.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("written:")
for p in sorted(OUT_DIR.iterdir()): print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")

GATE REPORT  (1 session(s), 137 gate calls, latest result per gate shown)

                                                                                         gate       scope status                                                                                                                                                                                                     detail              session
                                                                               CUDA available         env   PASS                                                                                                                                                                                                            2026-11-05T10-07-19Z
                                                                        torch is 2.10.0+cu128         env   PASS                                                                                                                                           